# Lab 3 天花板與地板

Lab 2 畫出了台積電的 K 線和 20 日均線。這個 Lab 用 Lab 1 寫的 `bias` 算出每一天的乖離率，從過去 5 年找出台積電的**天花板**和**地板**，再判斷今天的股價在哪裡。

> 規則來源：CMoney「權證小哥－地板天花板反轉價位監控表」官方產品頁公開的算法。
> 資料：Yahoo 財經（yfinance），抓到你執行的那一天。天花板、地板用「過去 5 年」算，所以每天執行的數字都會差一點；10/30 的均線和乖離率不會變。

## 第 0 步：裝好指定版本的套件

跟 Lab 2 一樣，先把套件釘成講義用的版本。**這一格一定要最先跑。**

> 預期：幾秒後格子左邊出現綠色勾勾，下面沒有任何訊息

In [ ]:
%pip install -q yfinance==0.2.66 plotly==5.24.1 pandas==2.2.3
# 💡 %pip install＝安裝套件；== 後面是版本；-q＝安靜模式，少印一點訊息

### 確認版本

> 預期：`0.2.66`，下一行 `5.24.1`

In [ ]:
import yfinance as yf
import plotly
print(yf.__version__)
print(plotly.__version__)

## D0・重新抓資料

跟 Lab 2 做的事一樣，這裡一格做完：抓資料、攤平表頭、刪掉成交量 0 的日子、算 20 日均線。

> 預期：最後 5 天的表格，最右邊多一欄 `MA20`

In [ ]:
stock_id = "2330"
df = yf.download(stock_id + ".TW", period="5y", auto_adjust=False)
df.columns = df.columns.droplevel(1)          # 💡 Lab 2 B1-4：攤平兩層表頭
df = df[df["Volume"] > 0]                     # 💡 Lab 2 B2-3：刪掉成交量 0 的假交易日
df["MA20"] = df["Close"].rolling(20).mean()   # 💡 Lab 2 B3-3：20 日均線
df.tail()

## D1・乖離率：每一天離均線多遠

乖離率＝(價格 − 均線) ÷ 均線 × 100

### D1-1・把 Lab 1 的乖離率函式拿回來

跟 Lab 1 A5-1 一模一樣。

> 這一格跑完不會有輸出，下一格才開始用它。

In [ ]:
___ bias(price, ma):                          # TODO：「定義一個新工具」的關鍵字
    return (_________) / _________ * 100      # TODO：價格比均線多多少？再除以誰？（跟 Lab 1 一字不改）

### D1-2・一次算出每一天的乖離率

> 預期：一整欄乖離率；最前面幾天是 `NaN`，最後幾天是正負幾個百分點的數字

In [ ]:
df["BIAS"] = bias(df[________], df[________])   # TODO：第一個放收盤價那一欄，第二個放 20 日均線那一欄
df["BIAS"]       # 💡 NaN＝空值：最前面 19 天還不滿 20 天，算不出 20 日均線，乖離率也就算不出來

## D2・天花板：正乖離排在第 95%

官方規則：
1. 找出所有**正乖離**的日子（股價 > 20 日均線）
2. 把它們的乖離率由小排到大
3. 排在第 95% 的那個值就是天花板（例：100 筆資料，取第 95 筆）

### D2-1・只留正乖離的日子

> 預期：700 多天，跟你執行的日期有關

In [ ]:
pos = df["BIAS"][__________ ]      # TODO：Lab 2 的「中括號放條件」：條件是「乖離率大於 0」
len(pos)                           # 💡 最前面 19 天還算不出 20 日均線，乖離率是空的，不會被算進來

### D2-2・天花板：第 95%

> 預期：一個 10 左右的數字（%），跟你執行的日期有關

In [ ]:
ceiling = pos.___________(0.95)   # TODO：「由小排到大，取排在第幾 % 的值」的工具（英文：分位數）
print(ceiling)                    # 💡 官方是數到第 95 筆；這個工具會在兩筆之間取中間值，所以會差一點點

## D3・地板：負乖離排在第 5%

官方規則：找出所有**負乖離**的日子（股價 < 20 日均線），由小排到大，排在第 5% 的就是地板。

### D3-1・只留負乖離的日子

> 預期：400 多天，跟你執行的日期有關

In [ ]:
neg = df["BIAS"][ _________ ]      # TODO：這次的條件是「乖離率小於 0」
len(neg)

### D3-2・地板：第 5%

> 預期：一個 -8 左右的數字（%），跟你執行的日期有關

In [ ]:
floor = neg.quantile(_________)   # TODO：第 5% 要寫成什麼數字？（看上面天花板的 95% 怎麼寫）
print(floor)                      # 💡 負數由小排到大，最前面是跌最深的；第 5% 就是「很少跌得比這更深」

## E1・換回價格，畫上 K 線

天花板和地板現在是百分比，要換回價格才能畫在 K 線上。把乖離率的公式倒過來：

價格 ＝ 均線 × (1 ＋ 乖離率 ÷ 100)

### E1-1・每一天的天花板價、地板價

> 預期：最後 5 天的 `Close`、`MA20`、`CEIL`、`FLOOR` 四欄

In [ ]:
df["CEIL"] = df["MA20"] * (___________)    # TODO：照上面倒過來的公式，乖離率用 ceiling
df["FLOOR"] = df["MA20"] * (___________)   # TODO：同一個公式，乖離率用 floor
df[["Close", "MA20", "CEIL", "FLOOR"]].tail()

### E1-2・畫出來

> 預期：K 線圖上有三條線：天花板（紅虛線）、20 日均線（橘）、地板（綠虛線）

In [ ]:
import plotly.graph_objects as go

recent = df.tail(120)
fig = go.Figure(go.Candlestick(
    x=recent.index,
    open=recent["Open"],
    high=recent["High"],
    low=recent["Low"],
    close=recent["Close"],
    increasing_line_color="red",
    decreasing_line_color="green",
    name="日K",
))
fig.add_trace(go.Scatter(x=recent.index, y=recent["CEIL"], name="天花板",
                         line=dict(color="red", dash="dash")))     # 💡 dash＝虛線
fig.add_trace(go.Scatter(x=recent.index, y=recent["MA20"], name="20日均線",
                         line=dict(color="orange")))
fig.add_trace(go.Scatter(x=recent.index, y=recent["FLOOR"], name="地板",
                         line=dict(color="green", dash="dash")))
fig.update_layout(title="台積電（2330）天花板與地板", xaxis_rangeslider_visible=False)
fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])
fig.show()

## E2・跌破地板的日子

### E2-1・5 年來收盤跌破地板的日子

> 預期：一張表，每一列是收盤價低於地板價的日子；有幾天跟你執行的日期有關

In [ ]:
below = df[df["Close"] < df["FLOOR"]]   # 💡 兩欄直接比：每一天的收盤價和同一天的地板價比
below[["Close", "FLOOR"]]

### E2-2・在圖上標出來

> 預期：上一張圖多了藍色圓點，標在最近半年跌破地板的日子

In [ ]:
below_recent = recent[recent["Close"] < recent["FLOOR"]]
fig.add_trace(go.Scatter(x=below_recent.index, y=below_recent["Close"], name="跌破地板",
                         mode="markers", marker=dict(color="blue", size=10)))   # 💡 markers＝只畫點、不連線
fig.show()

## E3・今天在哪裡？

### E3-1・最新一天

> 預期：最新交易日的日期和收盤價

In [ ]:
last = df.index[ _____ ]         # TODO：最後一個（最新那天）的位置，Lab 1 A2-3 用過
close = df.loc[last, "Close"]
print(last.date(), close)        # 💡 date()＝只留日期，不要後面的時間

### E3-2・用 Lab 1 的 if / elif 判斷

> 預期：在天花板之上、在地板之下、在中間，三種之一

In [ ]:
ceiling_price = df.loc[last, "CEIL"]
floor_price = df.loc[last, "FLOOR"]
if ______________:               # TODO：今天收盤有沒有高過天花板價？
    print("在天花板之上")
elif _______________:            # TODO：今天收盤有沒有低過地板價？
    print("在地板之下")
else:
    print("在中間")

## E4・今天的四個數字

把今天的狀態整理成四個數字，一眼看出股價離天花板、地板還有多遠。

> 預期：日期，加上 20 日均線、乖離率、天花板價、地板價四個數字

In [ ]:
print("日期", last.date())
print("20日均線", round(df.loc[last, "MA20"], 2))
print("乖離率", round(df.loc[last, "BIAS"], 2))
print("天花板價", round(ceiling_price, 1))
print("地板價", round(floor_price, 1))

## E5・匯出 CSV，帶回家

把今天算好的整張表存下來，回家用 Excel 也打得開。

### E5-1・存成 CSV 檔

> 這一格跑完不會有輸出。點 Colab 左邊的資料夾圖示，會看到 `2330_天花板地板.csv`

In [ ]:
df.________(stock_id + "_天花板地板.csv")   # TODO：把資料表「存成 CSV」的工具；CSV＝用逗號分隔的表格檔

### E5-2・下載到自己的電腦

> 預期：瀏覽器跳出下載，存到你的電腦。⚠️ Colab 關掉後，左邊資料夾裡的檔案會消失

In [ ]:
from google.colab import files
files.download(stock_id + "_天花板地板.csv")

### ✏️ 小作業：換一檔股票，再改改參數

**基本**：挑一檔自己的股票，算出它今天的四個數字。上市股票代號後面加 `.TW`，上櫃股票加 `.TWO`。

**進階挑戰**：權證小哥的規則是 20 日均線、第 95%／第 5%。難道只能這樣嗎？改改最上面的 `ma_days`、`upper`、`lower`，看四個數字和「5 年跌破地板天數」怎麼變。⚠️ 參數一改，就不是官方的規則了。

想一想：門檻從 0.05 改成 0.10，跌破地板的天數會怎樣？為什麼？

<details><summary>📖 做完再看：參考解與想一想的答案（參考解不只一種）</summary>

```python
stock_id = "2317"   # 基本：換一檔股票（上市加 .TW；上櫃要把下面的 .TW 改成 .TWO）
ma_days = 20        # 進階：試試 10、60
upper = 0.95        # 進階：試試 0.90、0.99
lower = 0.05        # 進階：試試 0.10、0.01

df = yf.download(stock_id + ".TW", period="5y", auto_adjust=False)
df.columns = df.columns.droplevel(1)
df = df[df["Volume"] > 0]
df["MA"] = df["Close"].rolling(ma_days).mean()
df["BIAS"] = bias(df["Close"], df["MA"])

ceiling = df["BIAS"][df["BIAS"] > 0].quantile(upper)
floor = df["BIAS"][df["BIAS"] < 0].quantile(lower)

last = df.index[-1]
ma = df.loc[last, "MA"]
print("日期", last.date())
print(str(ma_days) + "日均線", round(ma, 2))
print("乖離率", round(df.loc[last, "BIAS"], 2))
print("天花板價", round(ma * (1 + ceiling / 100), 1))
print("地板價", round(ma * (1 + floor / 100), 1))
print("5 年跌破地板天數", len(df[df["BIAS"] < floor]))
```

**想一想**：天數大約變成兩倍。地板的定義就是「負乖離裡最深的 5%」，改成 10%，跌破的日子自然多一倍左右。跌破幾天是你選的門檻決定的，不是市場決定的；「95%」是定義出來的，不是預測。

均線天數改長（例如 60 日），天花板、地板離均線的百分比會變大，但跌破地板的天數變化不大，因為每種均線都是跟自己的歷史比。
</details>

In [ ]:
# ✏️ 小作業：你的答案
stock_id = _______  # 基本：換一檔股票（上市加 .TW；上櫃要把下面的 .TW 改成 .TWO）
ma_days = 20        # 進階：試試 10、60
upper = 0.95        # 進階：試試 0.90、0.99
lower = 0.05        # 進階：試試 0.10、0.01

df = yf.download(stock_id + ".TW", period="5y", auto_adjust=False)
df.columns = df.columns.droplevel(1)
df = df[df["Volume"] > 0]
df["MA"] = df["Close"].rolling(ma_days).mean()
df["BIAS"] = bias(df["Close"], df["MA"])

ceiling = df["BIAS"][df["BIAS"] > 0].quantile(______)   # TODO：用上面的參數
floor = df["BIAS"][df["BIAS"] < 0].quantile(______)   # TODO：用上面的參數

last = df.index[-1]
ma = df.loc[last, "MA"]
print("日期", last.date())
print(str(ma_days) + "日均線", round(ma, 2))
print("乖離率", round(df.loc[last, "BIAS"], 2))
print("天花板價", round(ma * (1 + ceiling / 100), 1))
print("地板價", round(ma * (1 + floor / 100), 1))
print("5 年跌破地板天數", len(df[df["BIAS"] < floor]))

## 今天算出來的數字

| 寫法 | 做了什麼 |
|---|---|
| `bias(df["Close"], df["MA20"])` | Lab 1 的函式，一次算出每一天的乖離率 |
| `quantile(0.95)`、`quantile(0.05)` | 正乖離的第 95%＝天花板、負乖離的第 5%＝地板 |
| `MA20 * (1 + ... / 100)` | 換回每一天的天花板價、地板價 |
| `if` / `elif` | 判斷今天在天花板之上、地板之下、還是中間 |
| `to_csv` | 把整張表存成 CSV，帶回家 |

⚠️ 天花板和地板是用**同一段歷史**定義出來的：「5 年來只有 5% 的日子跌得比地板深」是在描述過去，不保證明天會反彈。